# NightmareNet Demo 🧠💤

**A Sleep-Inspired Training Paradigm for AI**

This notebook demonstrates the end-to-end NightmareNet pipeline:
1. Load a base dataset
2. Generate dream (mild distortion) and nightmare (extreme perturbation) data
3. Visualize distortion examples
4. Run one training cycle
5. Evaluate and compare metrics

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -e ..

In [ ]:
import random
import sys
sys.path.insert(0, '..')

import yaml
from datasets import Dataset

from nightmarenet.distortions.text import apply_text_distortions
from nightmarenet.distortions.semantic import apply_semantic_distortions
from nightmarenet.distortions.adversarial import apply_adversarial_distortions
from nightmarenet.data.generator import DreamDatasetGenerator, NightmareDatasetGenerator
from nightmarenet.training.scheduler import CyclicScheduler

random.seed(42)
print('NightmareNet loaded successfully!')

## 1. Sample Dataset

We'll create a small sample dataset to demonstrate the distortion pipeline.

In [ ]:
sample_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence that enables computers to learn.",
    "Paris is the capital of France and one of the most visited cities in the world.",
    "Deep learning uses neural networks with many layers to learn complex patterns.",
    "Natural language processing enables computers to understand and generate human language.",
    "The stock market experienced significant volatility during the past quarter.",
    "Climate change is one of the most pressing challenges facing humanity today.",
    "The new smartphone features an improved camera system and longer battery life.",
]

dataset = Dataset.from_dict({'text': sample_texts})
print(f'Dataset size: {len(dataset)} samples')
print(f'Sample: {dataset[0]["text"]}')

## 2. Distortion Examples

Let's see how each type of distortion affects the text.

In [ ]:
text = "Machine learning is a subset of artificial intelligence that enables computers to learn from data."

print('=' * 80)
print('ORIGINAL:')
print(text)
print()

# Dream-level distortions (mild)
random.seed(42)
dream_text = apply_text_distortions(text, strength=0.25)
print('DREAM (text distortion, strength=0.25):')
print(dream_text)
print()

random.seed(42)
dream_semantic = apply_semantic_distortions(text, strength=0.25)
print('DREAM (semantic distortion, strength=0.25):')
print(dream_semantic)
print()

# Nightmare-level distortions (extreme)
random.seed(42)
nightmare_text = apply_text_distortions(text, strength=0.8)
print('NIGHTMARE (text distortion, strength=0.8):')
print(nightmare_text)
print()

random.seed(42)
nightmare_semantic = apply_semantic_distortions(text, strength=0.8)
print('NIGHTMARE (semantic distortion, strength=0.8):')
print(nightmare_semantic)
print()

random.seed(42)
nightmare_adversarial = apply_adversarial_distortions(text, strength=0.8)
print('NIGHTMARE (adversarial distortion, strength=0.8):')
print(nightmare_adversarial)
print('=' * 80)

## 3. Generate Dream & Nightmare Datasets

Using the generator classes to produce full distorted datasets.

In [ ]:
# Generate dream data
dream_gen = DreamDatasetGenerator(strength=0.25, seed=42)
dream_data = dream_gen.generate(dataset)

# Generate nightmare data
nightmare_gen = NightmareDatasetGenerator(strength=0.8, seed=42)
nightmare_data = nightmare_gen.generate(dataset)

# Compare side by side
print('Side-by-side comparison:')
print('=' * 80)
for i in range(min(3, len(dataset))):
    print(f'\n--- Sample {i+1} ---')
    print(f'ORIGINAL:  {dataset[i]["text"]}')
    print(f'DREAM:     {dream_data[i]["text"]}')
    print(f'NIGHTMARE: {nightmare_data[i]["text"]}')

## 4. Training Schedule

View the training schedule that alternates between phases.

In [ ]:
scheduler = CyclicScheduler(
    num_cycles=3,
    wake_epochs=3,
    dream_epochs=2,
    nightmare_epochs=1,
    compression_rounds=1,
)

print(scheduler.summary())
print(f'\nTotal phase executions: {len(scheduler)}')

## 5. Full Training Pipeline (Conceptual)

To run the full pipeline with a real model:

```bash
python scripts/train.py --config configs/default.yaml
```

Or evaluate a checkpoint:

```bash
python scripts/evaluate.py --checkpoint checkpoints/final --config configs/default.yaml
```

In [ ]:
# Load config
with open('../configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Training Configuration:')
for section, values in config.items():
    if isinstance(values, dict):
        print(f'\n  {section}:')
        for k, v in values.items():
            print(f'    {k}: {v}')
    else:
        print(f'  {section}: {values}')

## Summary

NightmareNet introduces a biologically-inspired training paradigm:

1. **Wake Phase**: Standard training on clean data
2. **Dream Phase**: Training on mildly distorted data → encourages abstraction
3. **Nightmare Phase**: Training on extreme perturbations → stress-tests robustness
4. **Compression Phase**: Pruning/bottleneck → forces information efficiency

> *"We give AI a sleep cycle—so it learns what to forget, not just what to remember."*